# CITD Image Processing — Colab/Kaggle Pipeline

This notebook is intentionally a thin orchestrator. It clones the repository, installs the declared dependencies, loads the Roboflow secret, prepares the dataset at runtime, trains YOLO through `scripts/train-yolo.py`, and runs the existing inference/evaluation commands.

No model, dataset, or pipeline logic is duplicated here. Set `LPR_REPO_REF=develop` after the PR is merged; the default ref is the feature branch used to validate this notebook.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("LPR_REPO_URL", "https://github.com/cuongmn2011/CITD_ImageProcessing.git")
REPO_REF = os.getenv("LPR_REPO_REF", "feature/dataset-roboflow")

if Path("/content").is_dir():
    WORK_ROOT = Path("/content")
elif Path("/kaggle/working").is_dir():
    WORK_ROOT = Path("/kaggle/working")
else:
    WORK_ROOT = Path.cwd()
REPO_DIR = WORK_ROOT / "CITD_ImageProcessing"

def run(command, *, cwd=REPO_DIR, env=None):
    print("$", " ".join(str(part) for part in command))
    completed = subprocess.run(command, cwd=cwd, env=env, check=True, text=True)
    return completed

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], cwd=WORK_ROOT)
else:
    run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR)
    run(["git", "checkout", REPO_REF], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR)

print("Repository:", REPO_DIR)
print("Revision:", REPO_REF)

In [ ]:
# Install the project's declared environment; all later commands run through uv.
run([sys.executable, "-m", "pip", "install", "-q", "uv"], cwd=REPO_DIR)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv executable was not found after installation")
run([UV, "sync", "--extra", "vision", "--extra", "dataset", "--extra", "ocr"])


In [ ]:
def load_roboflow_secret():
    value = os.getenv("ROBOFLOW_API_KEY")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception:
        return None

ROBOFLOW_API_KEY = load_roboflow_secret()
if not ROBOFLOW_API_KEY:
    raise RuntimeError("Add ROBOFLOW_API_KEY to Colab userdata, Kaggle Secrets, or the environment")

ENV = os.environ.copy()
ENV["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
print("Roboflow secret loaded without printing its value.")

In [ ]:
DATASET_SPEC = os.getenv("LPR_ROBOFLOW_DATASET", "cuong-ta-ulxex/vietnamese-car-license-plate/1")
EPOCHS = int(os.getenv("LPR_EPOCHS", "50"))
IMAGE_SIZE = int(os.getenv("LPR_IMAGE_SIZE", "640"))
BATCH_SIZE = os.getenv("LPR_BATCH_SIZE", "-1")
DEVICE = os.getenv("LPR_DEVICE", "0")
print({"dataset": DATASET_SPEC, "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE, "device": DEVICE})

In [ ]:
# Download only when the local runtime cache is missing or invalid.
run([UV, "run", "--extra", "dataset", "python", "scripts/prepare-dataset.py", "--dataset", DATASET_SPEC], env=ENV)

In [ ]:
# Train with the repository's training script.
run([
    UV, "run", "--extra", "vision", "--extra", "dataset",
    "python", "scripts/train-yolo.py",
    "--dataset", DATASET_SPEC,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMAGE_SIZE),
    "--batch", BATCH_SIZE,
    "--device", DEVICE,
], env=ENV)

MODEL_PATH = REPO_DIR / "runs/detect/train/weights/best.pt"
if not MODEL_PATH.is_file():
    candidates = sorted((REPO_DIR / "runs").glob("**/weights/best.pt"))
    if not candidates:
        raise FileNotFoundError("Training completed without producing weights/best.pt")
    MODEL_PATH = candidates[-1]
print("Model:", MODEL_PATH)

In [ ]:
# Tesseract is an OS package, not only a Python dependency.
if shutil.which("tesseract") is None:
    run(["apt-get", "update", "-qq"], cwd=REPO_DIR)
    run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], cwd=REPO_DIR)
print("Tesseract:", shutil.which("tesseract"))

In [ ]:
# Run the existing image pipeline on one validation image.
configured_image = os.getenv("LPR_INPUT_IMAGE")
if configured_image:
    sample_image = Path(configured_image).expanduser()
else:
    candidates = []
    for split in ("valid", "val", "test", "train"):
        candidates.extend(sorted((REPO_DIR / "data/processed/license-plates" / split / "images").glob("*")))
    sample_image = candidates[0] if candidates else None
if sample_image is None or not sample_image.is_file():
    raise FileNotFoundError("Set LPR_INPUT_IMAGE or provide a valid exported validation image")

run([
    UV, "run", "lpr", "infer-image",
    "--image", str(sample_image),
    "--model", str(MODEL_PATH),
    "--ocr", "tesseract",
    "--variants", "otsu,clahe",
], env=ENV)
print("Inference command completed; inspect the JSON output above.")

In [ ]:
# Optional: evaluate predictions generated into a CSV with ground_truth,prediction columns.
ocr_csv = os.getenv("LPR_OCR_CSV")
if ocr_csv:
    run([UV, "run", "lpr", "evaluate-ocr", "--csv", ocr_csv], env=ENV)
else:
    print("OCR CSV evaluation skipped; set LPR_OCR_CSV to enable it.")

## Reproducibility checklist

Record `LPR_REPO_REF`, `DATASET_SPEC`, epoch/image-size/batch/device settings, generated model path, and the output metrics in the final report. Do not commit runtime secrets, downloaded data, or generated model weights.